**Goal:** Show which systems are riskiest in which months.

In [50]:
import pandas as pd
import numpy as np

In [51]:
DATA_PATH = "/Users/mehakxoxo/Documents/spring_2026/data_practicum/ai-predictive-maintenance-capstone/data/raw/FMUCD.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

/var/folders/29/flsbf9x54_s07dtd6_r_mrvm0000gn/T/ipykernel_11819/3603972264.py:3: DtypeWarning: Columns (3,4,6,10,17,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Shape: (3731442, 38)


In [52]:

df_rp = df_rp.copy()   # once, after any filtering

In [53]:
df.columns

Index(['UniversityID', 'Country', 'State/Province', 'BuildingID',
       'BuildingName', 'Size', 'Type', 'BuiltYear',
       'FCI (facility condition index)', 'CRV (current replacement value)',
       'DMC (deferred maintenance cost)', 'SystemCode', 'SystemDescription',
       'SubsystemCode', 'SubsystemDescription', 'DescriptiveCode',
       'ComponentDescription', 'WOID', 'WODescription', 'WOPriority',
       'WOStartDate', 'WOEndDate', 'WODuration', 'PPM/UPM', 'LaborCost',
       'MaterialCost', 'OtherCost', 'TotalCost', 'LaborHours', 'MinTemp.(°C)',
       'MaxTemp.(°C)', 'Atmospheric pressure(hPa)', 'Humidity(%)',
       'WindSpeed(m/s)', 'WindDegree', 'Precipitation(mm)', 'Snow(mm)',
       'Cloudness(%)'],
      dtype='object')

In [54]:
print(df.shape)
df.head(2)

(3731442, 38)


,UniversityID,Country,State/Province,BuildingID,BuildingName,Size,Type,BuiltYear,FCI (facility condition index),CRV (current replacement value),...,LaborHours,MinTemp.(°C),MaxTemp.(°C),Atmospheric pressure(hPa),Humidity(%),WindSpeed(m/s),WindDegree,Precipitation(mm),Snow(mm),Cloudness(%)
0,1,Canada,Nova Scotia,A050,COBURG ROAD 6414,5529.0,Research,1942.0,0.786664,981590.0,...,3.0,-4.232917,0.299583,1023.291667,82.541667,2.379167,170.583333,0.0,0.00,87.583333
1,1,Canada,Nova Scotia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,-6.034583,-0.784167,998.125000,75.625000,4.229167,278.541667,0.0,0.11,77.500000


Keep only the columns needed for this feature

In [55]:
KEEP_COLS = [
    "WOID",
    "SystemCode",
    "SystemDescription",
    "WOStartDate",
    "WOEndDate",
    "PPM/UPM",
    # costs optional (keep if you want later)
    "TotalCost",
    # weather (optional but you already have them)
    "MinTemp.(°C)", "MaxTemp.(°C)", "Atmospheric pressure(hPa)", "Humidity(%)",
    "WindSpeed(m/s)", "WindDegree", "Precipitation(mm)", "Snow(mm)", "Cloudness(%)",
]

df_rp = df[[c for c in KEEP_COLS if c in df.columns]].copy()
print(df_rp.shape)
df_rp.head(2)

(3731442, 16)


,WOID,SystemCode,SystemDescription,WOStartDate,WOEndDate,PPM/UPM,TotalCost,MinTemp.(°C),MaxTemp.(°C),Atmospheric pressure(hPa),Humidity(%),WindSpeed(m/s),WindDegree,Precipitation(mm),Snow(mm),Cloudness(%)
0,WO244376,D20,Plumbing,2012-12-17 07:49:05,2015-12-07 12:41:28,PPM,175.8,-4.232917,0.299583,1023.291667,82.541667,2.379167,170.583333,0.0,0.00,87.583333
1,WO245713,D30,HVAC,2013-01-02 11:31:55,2017-03-22 10:51:59,UPM,291.5,-6.034583,-0.784167,998.125000,75.625000,4.229167,278.541667,0.0,0.11,77.500000


FM-Focused One-Line Summaries (Maintenance & Risk Oriented)

Plumbing – Water supply and drainage systems prone to leaks, clogs, and corrosion failures.

HVAC – Climate control systems with high failure risk from seasonal load and aging components.

Interior Finishes – Wear-and-tear surfaces affecting aesthetics more than system reliability.

Interior Construction – Interior partitions and assemblies impacting space functionality.

Fire Protection – Life-safety systems requiring strict inspection, testing, and compliance.

Electrical – Power and lighting systems with risks of outages, overloads, and faults.

Equipment – Installed operational equipment with usage-driven maintenance needs.

Furnishings – Non-critical movable assets with low maintenance risk.

General – Miscellaneous work orders not tied to a specific building system.

Conveying – Elevators and lifts with high safety and downtime impact.

Site Mechanical Utilities – External mechanical systems supporting campus operations.

Exterior Enclosure – Building envelope components exposed to weather-related degradation.

Stairs – Vertical circulation elements with safety and accessibility implications.

Roofing – Weather-exposed system with high leak and seasonal failure risk.

Unclassified – Work orders lacking sufficient system identification.

Site Improvements – Outdoor enhancements impacting usability rather than operations.

Special Construction – Specialized structures with unique maintenance profiles.

Superstructure – Load-bearing structural systems with long-term risk implications.

Foundations – Substructure elements critical to building stability.

Site Preparation – Early-phase ground work with minimal long-term maintenance impact.

Site Electrical Utilities – External power infrastructure supporting campus systems.

Selective Building Demolition – Targeted removals typically linked to renovation projects.

Other Site Construction – Site activities not aligned with standard FM categories.

NaN / Missing – Records requiring cleanup or reassignment during preprocessing.


Clean Attribute 1: WOID (unique key)


In [56]:
df_rp.WOID

0          WO244376
1          WO245713
2          WO246478
3          WO247428
4          WO248110
             ...   
3731437    13459035
3731438    13459037
3731439    13459036
3731440    13459038
3731441    13459039
Name: WOID, Length: 3731442, dtype: object

In [57]:
df_rp["WOID"].isnull().sum()

np.int64(6878)

In [58]:
print(df_rp["WOID"].dtypes)

object


In [59]:
df_rp["WOID"].unique()

array(['WO244376', 'WO245713', 'WO246478', ..., 13459036, 13459038,
       13459039], shape=(2662067,), dtype=object)

In [60]:
import pandas as pd
import numpy as np

df_rp = df.copy()

def miss_report(s, name):
    n = len(s)
    na = s.isna().sum()
    print(f"{name}: missing {na}/{n} ({na/n:.1%}) | dtype={s.dtype}")

def trim_text(s):
    # Keep NaN as NaN, strip whitespace, convert empty strings to NaN
    s = s.astype("string")
    s = s.str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return s


In [61]:
miss_report(df_rp["WOID"], "WOID")

# If WOID exists but has blanks/strings, clean it
df_rp["WOID"] = trim_text(df_rp["WOID"])

miss_report(df_rp["WOID"], "WOID (after trim)")

before = len(df_rp)
df_rp = df_rp.dropna(subset=["WOID"])
print("Dropped rows missing WOID:", before - len(df_rp))

before = len(df_rp)
df_rp = df_rp.drop_duplicates(subset=["WOID"])
print("Removed duplicate WOID rows:", before - len(df_rp))


WOID: missing 6878/3731442 (0.2%) | dtype=object
WOID (after trim): missing 6878/3731442 (0.2%) | dtype=string
Dropped rows missing WOID: 6878
Removed duplicate WOID rows: 1062505


In [62]:
df_rp["WOID"].isnull().sum()

np.int64(0)

Clean Attribute 2: WOStartDate (needed for Month)

Goal: parse to datetime; drop rows with invalid start date.

In [63]:
miss_report(df_rp["WOStartDate"], "WOStartDate")

df_rp["WOStartDate"] = pd.to_datetime(df_rp["WOStartDate"], errors="coerce")

miss_report(df_rp["WOStartDate"], "WOStartDate (parsed)")
before = len(df_rp)
df_rp = df_rp.dropna(subset=["WOStartDate"])
print("Dropped rows with invalid/missing WOStartDate:", before - len(df_rp))

print("Date range:", df_rp["WOStartDate"].min(), "→", df_rp["WOStartDate"].max())

WOStartDate: missing 0/2662059 (0.0%) | dtype=object
WOStartDate (parsed): missing 1014798/2662059 (38.1%) | dtype=datetime64[ns]
Dropped rows with invalid/missing WOStartDate: 1014798
Date range: 2002-09-06 12:09:12 → 2021-05-28 00:00:00


In [64]:
df_rp["WOStartDate"].isnull().sum()

np.int64(0)

Clean Attribute 3: SystemDescription (row label in heatmap)

Goal: standardize text; drop missing; optionally collapse rare systems later.

In [65]:
df_rp["SystemDescription"].isnull().sum()

np.int64(3)

In [66]:
miss_report(df_rp["SystemDescription"], "SystemDescription")

df_rp["SystemDescription"] = trim_text(df_rp["SystemDescription"])

miss_report(df_rp["SystemDescription"], "SystemDescription (after trim)")
before = len(df_rp)
df_rp = df_rp.dropna(subset=["SystemDescription"])
print("Dropped rows missing SystemDescription:", before - len(df_rp))

# optional: unify casing (pick one)
df_rp["SystemDescription"] = df_rp["SystemDescription"].str.title()

df_rp["SystemDescription"].value_counts().head(10)


SystemDescription: missing 3/1647261 (0.0%) | dtype=object
SystemDescription (after trim): missing 3/1647261 (0.0%) | dtype=string
Dropped rows missing SystemDescription: 3


SystemDescription
Hvac                     509995
Electrical               469784
Plumbing                 274540
Fire Protection          111640
Interior Construction     80163
Equipment                 52690
Exterior Enclosure        37233
Interior Finishes         26957
General                   23690
Furnishings               18236
Name: count, dtype: Int64

Clean Attribute 4: SystemCode (optional fallback)

Goal: clean text; do not drop rows if missing (because you already kept SystemDescription).

In [67]:
if "SystemCode" in df_rp.columns:
    miss_report(df_rp["SystemCode"], "SystemCode")
    df_rp["SystemCode"] = trim_text(df_rp["SystemCode"])
    miss_report(df_rp["SystemCode"], "SystemCode (after trim)")

SystemCode: missing 8570/1647258 (0.5%) | dtype=object
SystemCode (after trim): missing 8570/1647258 (0.5%) | dtype=string


Clean Attribute 5: PPM/UPM (target label)

This is the most important missing-value step: if it’s missing/unknown, you can’t use that row for historical risk or supervised ML.

A) Inspect raw values first

In [68]:
miss_report(df_rp["PPM/UPM"], "PPM/UPM")

df_rp["PPM/UPM"] = trim_text(df_rp["PPM/UPM"]).str.upper()

df_rp["PPM/UPM"].value_counts(dropna=False).head(30)

PPM/UPM: missing 8308/1647258 (0.5%) | dtype=object


PPM/UPM
PPM     1052835
UPM      586115
<NA>       8308
Name: count, dtype: Int64

B) Map to binary UPM and drop unmapped/missing

In [69]:
LABEL_MAP = {"UPM": 1, "PPM": 0}

df_rp["UPM"] = df_rp["PPM/UPM"].map(LABEL_MAP)

# Show unmapped values (so you can decide what to do)
unmapped = df_rp.loc[df_rp["UPM"].isna(), "PPM/UPM"].value_counts(dropna=False).head(30)
print("Unmapped label values:\n", unmapped)

before = len(df_rp)
df_rp = df_rp.dropna(subset=["UPM"])
df_rp["UPM"] = df_rp["UPM"].astype(int)
print("Dropped rows missing/unmapped PPM/UPM:", before - len(df_rp))

df_rp["UPM"].value_counts()

Unmapped label values:
 PPM/UPM
<NA>    8308
Name: count, dtype: Int64
Dropped rows missing/unmapped PPM/UPM: 8308


UPM
0    1052835
1     586115
Name: count, dtype: int64

In [70]:
df_rp["UPM"].isnull().sum()

np.int64(0)

Missing values for Weather Columns (optional features)

For weather, we usually don’t drop rows unless missing is extreme. We impute.

A) Convert to numeric

In [71]:
WEATHER_COLS = [
    "MinTemp.(°C)", "MaxTemp.(°C)", "Atmospheric pressure(hPa)", "Humidity(%)",
    "WindSpeed(m/s)", "WindDegree", "Precipitation(mm)", "Snow(mm)", "Cloudness(%)"
]

for c in WEATHER_COLS:
    if c in df_rp.columns:
        df_rp[c] = pd.to_numeric(df_rp[c], errors="coerce")
        miss_report(df_rp[c], c)

MinTemp.(°C): missing 0/1638950 (0.0%) | dtype=float64
MaxTemp.(°C): missing 0/1638950 (0.0%) | dtype=float64
Atmospheric pressure(hPa): missing 698164/1638950 (42.6%) | dtype=float64
Humidity(%): missing 0/1638950 (0.0%) | dtype=float64
WindSpeed(m/s): missing 0/1638950 (0.0%) | dtype=float64
WindDegree: missing 702610/1638950 (42.9%) | dtype=float64
Precipitation(mm): missing 0/1638950 (0.0%) | dtype=float64
Snow(mm): missing 0/1638950 (0.0%) | dtype=float64
Cloudness(%): missing 702610/1638950 (42.9%) | dtype=float64


In [72]:
for c in WEATHER_COLS:
    if c in df_rp.columns:
        med = df_rp[c].median()
        df_rp[c] = df_rp[c].fillna(med)

Create Month/Season after cleaning essentials

In [73]:
df_rp["MonthNum"] = df_rp["WOStartDate"].dt.month
df_rp["MonthName"] = df_rp["WOStartDate"].dt.strftime("%b")

def month_to_season(m):
    if m in [12, 1, 2]: return "Winter"
    if m in [3, 4, 5]: return "Spring"
    if m in [6, 7, 8]: return "Summer"
    return "Fall"

df_rp["Season"] = df_rp["MonthNum"].apply(month_to_season)

df_rp[["WOID","SystemDescription","WOStartDate","MonthName","PPM/UPM","UPM"]].head()

,WOID,SystemDescription,WOStartDate,MonthName,PPM/UPM,UPM
0,WO244376,Plumbing,2012-12-17 07:49:05,Dec,PPM,0
1,WO245713,Hvac,2013-01-02 11:31:55,Jan,UPM,1
2,WO246478,Hvac,2013-01-07 09:22:45,Jan,PPM,0
3,WO247428,Interior Finishes,2013-01-15 09:05:33,Jan,UPM,1
4,WO248110,Hvac,2013-01-21 09:42:39,Jan,UPM,1


In [74]:
df_rp["MonthName"].isnull().sum()

np.int64(0)

Quick final sanity check

In [75]:
print("Final rows:", len(df_rp))
print("Unique systems:", df_rp["SystemDescription"].nunique())
print("Overall UPM rate:", df_rp["UPM"].mean())

Final rows: 1638950
Unique systems: 23
Overall UPM rate: 0.3576161566856829
